# Reducing 30-Day Hospital Readmissions
## Using Predictive Risk Scoring and Intervention Insights

**Dataset:** Diabetes 130-US Hospitals (1999–2008) — UCI Machine Learning Repository  
**Objective:** Build a healthcare analytics system that identifies high-risk patients and suggests interventions to reduce readmissions.Cell 2 — Markdown:

In [55]:
!pip install pandas
!pip install numpy

In [56]:
import pandas as pd
import numpy as np
import sqlite3

In [57]:
df = pd.read_csv("diabetic_data.csv")
print(df.shape)
df.head()

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [58]:
print(df.columns.tolist())
print(f"\nTotal rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Total rows: 101766
Total columns: 50


In [59]:
for col in df.columns:
    missing = (df[col] == '?').sum()
    if missing > 0:
        print(f"{col}: {missing} missing")

race: 2273 missing
weight: 98569 missing
payer_code: 40256 missing
medical_specialty: 49949 missing
diag_1: 21 missing
diag_2: 358 missing
diag_3: 1423 missing


In [60]:
print(df['readmitted'].value_counts())

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


### Cleaning the data

In [61]:
#replace ? with NaN
df.replace('?', np.nan, inplace=True)
print(f"Total missing values: {df.isnull().sum().sum()}")

Total missing values: 374017


In [62]:
# <30 means readmitted within 30 days = 1, everything else = 0
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)
print(df['readmitted_30'].value_counts())
print(f"\nReadmission rate: {round(df['readmitted_30'].mean() * 100, 1)}%")

readmitted_30
0    90409
1    11357
Name: count, dtype: int64

Readmission rate: 11.2%


In [63]:
#dropping columns which are not needed. 
df.drop(columns=['encounter_id', 'patient_nbr', 'readmitted'], inplace=True)
print(f"Shape now: {df.shape}")

Shape now: (101766, 48)


In [64]:
# weight is ~97% missing, payer_code ~40%, medical_specialty ~49%
df.drop(columns=['weight', 'payer_code', 'medical_specialty'], inplace=True)
print(f"Shape now: {df.shape}")

Shape now: (101766, 45)


In [50]:
df['race']=df['race'].fillna('Unknown')
df['diag_1']=df['diag_1'].fillna('Unknown')
df['diag_2']=df['diag_2'].fillna('Unknown')
df['diag_2']=df['diag_3'].fillna('Unknown')

print(f"Missing values left: {df.isnull().sum().sum()}")

Missing values left: 181168


In [51]:
# Patients who died or went to hospice can't be readmitted — that's data leakage
leakage_ids = [11, 13, 14, 19, 20, 21]
before = len(df)
df = df[~df['discharge_disposition_id'].isin(leakage_ids)]
print(f"Removed {before - len(df)} rows (deceased/hospice)")
print(f"Final shape: {df.shape}")



Removed 2423 rows (deceased/hospice)
Final shape: (99343, 45)


### Store in SQLite

In [70]:
# Connect to SQLite and save our cleaned dataframe as a table called 'encounters'
conn = sqlite3.connect("hospital.db")
df.to_sql("encounters", conn, if_exists="replace", index=False)

# Verify the data was saved correctly
result = pd.read_sql("SELECT COUNT(*) as rows FROM encounters", conn)
print(f"Rows saved to database: {result['rows'][0]}")
conn.close()


Rows saved to database: 101766


In [92]:
# Install the SQL magic extension so we can write SQL directly in cells
!pip install ipython-sql sqlalchemy
!pip install prettytable==3.10.0

  Attempting uninstall: prettytable
    Found existing installation: prettytable 3.9.0
    Uninstalling prettytable-3.9.0:
      Successfully uninstalled prettytable-3.9.0


In [1]:
%load_ext sql
%sql sqlite:///hospital.db

### SQL Queries — Exploring the data

In [13]:
%%sql
SELECT age, COUNT(*) as total,
       ROUND(AVG(readmitted_30) * 100, 1) as readmission_rate
FROM encounters
GROUP BY age
ORDER BY readmission_rate DESC

 * sqlite:///hospital.db
Done.


age,total,readmission_rate
[20-30),1657,14.2
[80-90),17197,12.1
[70-80),26068,11.8
[30-40),3775,11.2
[90-100),2793,11.1
[60-70),22483,11.1
[40-50),9685,10.6
[50-60),17256,9.7
[10-20),691,5.8
[0-10),161,1.9


In [15]:
%%sql
SELECT number_inpatient, COUNT(*) as total,
       ROUND(AVG(readmitted_30) * 100, 1) as readmission_rate
FROM encounters
GROUP BY number_inpatient
ORDER BY number_inpatient

 * sqlite:///hospital.db
Done.


number_inpatient,total,readmission_rate
0,67630,8.4
1,19521,12.9
2,7566,17.4
3,3411,20.3
4,1622,23.6
5,812,31.4
6,480,34.6
7,268,35.4
8,151,44.4
9,111,42.3


In [16]:
%%sql
SELECT readmitted_30,
       ROUND(AVG(time_in_hospital), 2) as avg_stay,
       COUNT(*) as count
FROM encounters
GROUP BY readmitted_30

 * sqlite:///hospital.db
Done.


readmitted_30,avg_stay,count
0,4.35,90409
1,4.77,11357
